In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

clinical_gex = pd.read_csv('../dataset/created/clinical_gex/clinical_gex_rfe.csv')
y = clinical_gex['pfs_label']
x = clinical_gex.drop(['pfs_label', 'Unnamed: 0'], axis=1)
x.head()

,bor_prep__BOR,remainder__CITED1,remainder__IL1RAPL1,remainder__S100A8,remainder__EGFL8,remainder__CDH2,remainder__FAM178B,remainder__SCRG1,remainder__BAIAP2L1,remainder__PAEP,...,remainder__CD79A,remainder__PLA1A,remainder__HMOX1,remainder__MIA,remainder__TFAP2A,remainder__MAGEA12,remainder__HMCN1,remainder__EPDR1,remainder__TSPAN10,remainder__P2RX7
0,0.0,5.683933,4.974260,6.650094,4.738944,6.787626,5.209628,5.182510,5.403285,5.192435,...,6.127600,7.427885,6.577089,10.749602,7.635757,6.328177,6.602261,6.775539,7.142719,6.156618
1,3.0,7.201570,5.074990,6.139270,6.191980,6.110276,5.204983,5.138783,5.537916,5.249190,...,5.230650,7.243291,7.049003,11.568732,8.274490,6.510576,5.438397,6.658401,7.400251,6.761518
2,3.0,5.662868,5.765472,6.278684,4.705266,6.055845,5.207164,5.487081,5.343537,5.283647,...,5.863787,7.804214,6.582462,9.779468,6.916894,5.802156,5.674448,6.716163,7.052908,6.260890
3,0.0,5.852352,4.731015,7.390692,4.731072,5.953089,5.253117,5.164513,5.481515,5.162883,...,5.270042,5.674434,10.947866,6.762897,7.782930,5.084814,5.965363,6.727260,7.135489,6.352753
4,0.0,5.780264,5.068498,6.312233,6.407063,6.060600,5.205769,6.065713,5.492277,5.170959,...,4.923446,5.426332,6.673273,9.594480,10.490477,6.509635,7.621351,6.573656,7.201527,7.251177


In [6]:
import os
import shutil

import optuna

import sklearn.datasets
import sklearn.metrics
import xgboost as xgb

SEED = 108
N_FOLDS = 3
CV_RESULT_DIR = "../run-models/xgboost_cv_results"

In [7]:
def objective(trial):
    data, target = x, y
    dtrain = xgb.DMatrix(data, label=target)

    param = {
        "verbosity": 0,
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
        # sampling ratio for training data.
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        # sampling according to each tree.
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
    }

    if param["booster"] == "gbtree" or param["booster"] == "dart":
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)
        # minimum child weight, larger the term more conservative the tree.
        param["min_child_weight"] = trial.suggest_int("min_child_weight", 2, 10)
        param["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)
        param["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

    if param["booster"] == "dart":
        param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        param["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0, log=True)
        param["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0, log=True)

    xgb_cv_results = xgb.cv(
        params=param,
        dtrain=dtrain,
        num_boost_round=10000,
        nfold=N_FOLDS,
        stratified=True,
        early_stopping_rounds=100,
        seed=SEED,
        verbose_eval=False,
    )

    # Set n_estimators as a trial attribute; Accessible via study.trials_dataframe().
    trial.set_user_attr("n_estimators", len(xgb_cv_results))

    # Save cross-validation results.
    filepath = os.path.join(CV_RESULT_DIR, "{}.csv".format(trial.number))
    xgb_cv_results.to_csv(filepath, index=False)

    # Extract the best score.
    best_score = xgb_cv_results["test-mlogloss-mean"].values[-1]
    return best_score

In [9]:
if not os.path.exists(CV_RESULT_DIR):
    os.mkdir(CV_RESULT_DIR)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, timeout=600)

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

print("  Number of estimators: {}".format(trial.user_attrs["n_estimators"]))

shutil.rmtree(CV_RESULT_DIR)

[I 2026-03-24 16:39:16,412] A new study created in memory with name: no-name-b7c07199-be98-48b3-a5ff-e50154bb864e
[I 2026-03-24 16:39:16,640] Trial 0 finished with value: 0.4258936840441511 and parameters: {'booster': 'gblinear', 'lambda': 6.488369378599357e-08, 'alpha': 0.00011388784690438003, 'subsample': 0.2124294169720626, 'colsample_bytree': 0.3108686699247336}. Best is trial 0 with value: 0.4258936840441511.
[W 2026-03-24 16:39:17,796] Trial 1 failed with parameters: {'booster': 'dart', 'lambda': 0.0027176599245337775, 'alpha': 5.582819518350245e-07, 'subsample': 0.5176085176526339, 'colsample_bytree': 0.3776429350013477, 'max_depth': 6, 'min_child_weight': 2, 'eta': 0.002407264125802613, 'gamma': 0.3666111576823097, 'grow_policy': 'depthwise', 'sample_type': 'weighted', 'normalize_type': 'tree', 'rate_drop': 2.7145164602286352e-05, 'skip_drop': 0.0362460910345429} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/buudinhha/Pyc

KeyboardInterrupt: 